# 27 · Overlap Phase Diagram and Invariant Region

This notebook turns the multiscale bootstrap results into a **phase-style map**.

Notebook 26 showed:
- zeta vs GUE has persistently higher overlap than the Poisson tasks
- that pattern is bootstrap-stable across window size and height

Now we ask:

> Can that pattern be summarized as a stable overlap phase region?

## Tasks

We compare:
1. **zeta vs GUE**
2. **zeta vs Poisson**
3. **GUE vs Poisson**

## Main outputs

- mean overlap phase diagrams
- overlap-gap phase diagrams
- height-resolved overlap diagrams
- bootstrap-certainty masks
- a compact invariant-region summary

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 2200
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        all_gaps.extend(np.diff(bulk_vals).tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Descriptor builders

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def ratio_std(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.std(r))

def build_feature_matrix(gaps, feature_family="minimal", k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])

    if feature_family == "ratio_only":
        X = np.array([[ratio_mean(w), ratio_std(w)] for w in windows_n], dtype=float)
    elif feature_family == "minimal":
        X = np.array([[window_entropy(w), unevenness(w), ratio_mean(w)] for w in windows_n], dtype=float)
    elif feature_family == "full":
        X = np.array([[window_entropy(w), unevenness(w), reversal_symmetry_score(w), ratio_mean(w), ratio_std(w)] for w in windows_n], dtype=float)
    elif feature_family == "raw_window":
        X = windows_n.copy()
    else:
        raise ValueError(f"Unknown feature family: {feature_family}")

    return windows_n, X

## RBF bootstrap evaluator

In [ ]:
def split_train_test(X, frac=0.6):
    n = len(X)
    cut = int(frac * n)
    return X[:cut], X[cut:]

def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad
    return w

def decision_function_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ w

def predict_proba_logistic(X, w):
    return sigmoid(decision_function_logistic(X, w))

def choose_prototypes(X, n_proto=20):
    idx = np.linspace(0, len(X) - 1, n_proto).astype(int)
    return X[idx]

def estimate_rbf_gamma(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    med = np.median(D[D > 0])
    if med <= 0:
        med = 1.0
    return 1.0 / (2.0 * med * med)

def rbf_features(X, prototypes, gamma):
    D2 = np.sum((X[:, None, :] - prototypes[None, :, :])**2, axis=2)
    return np.exp(-gamma * D2)

def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

def overlap_coefficient_from_hist(a, b, bins=30):
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    if hi <= lo:
        return 1.0
    hist_a, edges = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hist_b, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    dx = edges[1] - edges[0]
    return float(np.sum(np.minimum(hist_a, hist_b)) * dx)

def evaluate_pair_rbf(X0, X1):
    m = min(len(X0), len(X1))
    X0 = X0[:m]
    X1 = X1[:m]

    X0_train, X0_test = split_train_test(X0, frac=0.6)
    X1_train, X1_test = split_train_test(X1, frac=0.6)

    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))

    X_test = np.vstack([X0_test, X1_test])
    y_test = np.array([0] * len(X0_test) + [1] * len(X1_test))

    Xtr_std, Xte_std = standardize_train_test(X_train, X_test)
    prototypes = choose_prototypes(Xtr_std, n_proto=min(20, len(Xtr_std)))
    gamma = estimate_rbf_gamma(Xtr_std)
    R_train = rbf_features(Xtr_std, prototypes, gamma)
    R_test = rbf_features(Xte_std, prototypes, gamma)

    w = fit_logistic_regression(R_train, y_train, lr=0.15, n_steps=3500, reg=1e-4)
    scores = decision_function_logistic(R_test, w)
    probs = predict_proba_logistic(R_test, w)
    preds = (probs >= 0.5).astype(int)

    acc = accuracy(y_test, preds)
    scores0 = scores[y_test == 0]
    scores1 = scores[y_test == 1]

    return {
        "accuracy": acc,
        "score_overlap": overlap_coefficient_from_hist(scores0, scores1, bins=30),
    }

def bootstrap_rows(X, rng):
    idx = rng.integers(0, len(X), size=len(X))
    return X[idx]

def bootstrap_pair_rbf(X0, X1, n_boot=40, seed=9423):
    boot_rng = np.random.default_rng(seed)

    acc_vals = []
    score_vals = []

    m = min(len(X0), len(X1))
    X0 = X0[:m]
    X1 = X1[:m]

    for _ in range(n_boot):
        X0b = bootstrap_rows(X0, boot_rng)
        X1b = bootstrap_rows(X1, boot_rng)
        out = evaluate_pair_rbf(X0b, X1b)
        acc_vals.append(out["accuracy"])
        score_vals.append(out["score_overlap"])

    acc_vals = np.array(acc_vals)
    score_vals = np.array(score_vals)

    return {
        "acc_mean": float(acc_vals.mean()),
        "acc_std": float(acc_vals.std()),
        "acc_q025": float(np.quantile(acc_vals, 0.025)),
        "acc_q975": float(np.quantile(acc_vals, 0.975)),
        "score_mean": float(score_vals.mean()),
        "score_std": float(score_vals.std()),
        "score_q025": float(np.quantile(score_vals, 0.025)),
        "score_q975": float(np.quantile(score_vals, 0.975)),
    }

## Reduced phase-diagram grid

In [ ]:
window_sizes = [3, 5, 7]
feature_families = ["ratio_only", "minimal", "full", "raw_window"]
feature_complexity = {"ratio_only": 1, "minimal": 2, "full": 3, "raw_window": 4}
height_blocks = [(0, 400), (400, 800), (800, 1200), (1200, 1600)]
requested_sample_sizes = [200, 400]
n_boot = 40

## Main bootstrap sweep

In [ ]:
phase_results = []

for k in window_sizes:
    for feature_family in feature_families:
        for (start, stop) in height_blocks:
            zeta_g = zeta_gaps_full[start:stop]
            poisson_g = poisson_gaps_full[start:stop]
            gue_needed = max(stop - start + 300, 900)
            gue_g = gue_gaps_full[:gue_needed]

            _, zeta_X_all = build_feature_matrix(zeta_g, feature_family=feature_family, k=k)
            _, poisson_X_all = build_feature_matrix(poisson_g, feature_family=feature_family, k=k)
            _, gue_X_all = build_feature_matrix(gue_g, feature_family=feature_family, k=k)

            max_available = min(len(zeta_X_all), len(poisson_X_all), len(gue_X_all))
            available_ns = [n for n in requested_sample_sizes if n <= max_available]
            if not available_ns:
                if max_available >= 40:
                    available_ns = [max_available]
                else:
                    continue

            for n in available_ns:
                zeta_X = zeta_X_all[:n]
                poisson_X = poisson_X_all[:n]
                gue_X = gue_X_all[:n]

                for i, (task_name, A, B) in enumerate([
                    ("zeta_vs_GUE", zeta_X, gue_X),
                    ("zeta_vs_Poisson", zeta_X, poisson_X),
                    ("GUE_vs_Poisson", gue_X, poisson_X),
                ]):
                    out = bootstrap_pair_rbf(A, B, n_boot=n_boot, seed=9423 + 1000*i + 10*k + n + start)
                    phase_results.append({
                        "task": task_name,
                        "k": k,
                        "feature_family": feature_family,
                        "feature_complexity": feature_complexity[feature_family],
                        "height_block": f"{start+1}-{stop}",
                        "sample_size": n,
                        "max_available": max_available,
                        **out,
                    })

len(phase_results)

## Quick diagnostic

In [ ]:
sorted(set(r["sample_size"] for r in phase_results)), sorted(set(r["feature_family"] for r in phase_results)), sorted(set(r["height_block"] for r in phase_results))

## Helpers

In [ ]:
def nearest_available_n(records, task=None, fixed_filter=None, target_n=400):
    vals = []
    for r in records:
        if task is not None and r["task"] != task:
            continue
        if fixed_filter is not None:
            ok = True
            for k, v in fixed_filter.items():
                if r.get(k) != v:
                    ok = False
                    break
            if not ok:
                continue
        vals.append(r["sample_size"])
    if not vals:
        return None
    unique_vals = sorted(set(vals))
    return min(unique_vals, key=lambda x: abs(x - target_n))

def build_feature_k_matrix(records, task, metric_name, target_n=400):
    actual_n = nearest_available_n(records, task=task, target_n=target_n)
    M = np.full((len(feature_families), len(window_sizes)), np.nan)

    for i, ff in enumerate(feature_families):
        for j, k in enumerate(window_sizes):
            vals = [
                r[metric_name] for r in records
                if r["task"] == task and r["feature_family"] == ff and r["k"] == k and r["sample_size"] == actual_n
            ]
            if vals:
                M[i, j] = float(np.mean(vals))
    return M, actual_n

def build_height_k_matrix(records, task, metric_name, fixed_feature="minimal", target_n=400):
    actual_n = nearest_available_n(records, task=task, fixed_filter={"feature_family": fixed_feature}, target_n=target_n)
    M = np.full((len(window_sizes), len(height_blocks)), np.nan)

    for i, k in enumerate(window_sizes):
        for j, (start, stop) in enumerate(height_blocks):
            label = f"{start+1}-{stop}"
            vals = [
                r[metric_name] for r in records
                if r["task"] == task and r["feature_family"] == fixed_feature
                and r["k"] == k and r["height_block"] == label and r["sample_size"] == actual_n
            ]
            if vals:
                M[i, j] = float(np.mean(vals))
    return M, actual_n

def aggregate_feature_slice(records, task, metric_prefix, fixed_k=5, target_n=400):
    actual_n = nearest_available_n(records, task=task, fixed_filter={"k": fixed_k}, target_n=target_n)
    xs, means, lows, highs = [], [], [], []
    for ff in feature_families:
        vals = [r for r in records if r["task"] == task and r["feature_family"] == ff and r["k"] == fixed_k and r["sample_size"] == actual_n]
        if vals:
            mean = float(np.mean([v[f"{metric_prefix}_mean"] for v in vals]))
            lo = float(np.mean([v[f"{metric_prefix}_mean"] - v[f"{metric_prefix}_q025"] for v in vals]))
            hi = float(np.mean([v[f"{metric_prefix}_q975"] - v[f"{metric_prefix}_mean"] for v in vals]))
            xs.append(ff)
            means.append(mean)
            lows.append(lo)
            highs.append(hi)
    return xs, np.array(means), np.array(lows), np.array(highs), actual_n

## Mean overlap phase diagrams (feature family × window size)

In [ ]:
M_zg, n_zg = build_feature_k_matrix(phase_results, "zeta_vs_GUE", "score_mean", target_n=400)
M_zp, n_zp = build_feature_k_matrix(phase_results, "zeta_vs_Poisson", "score_mean", target_n=400)
M_gp, n_gp = build_feature_k_matrix(phase_results, "GUE_vs_Poisson", "score_mean", target_n=400)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharey=True)

for ax, M, title, n_used in zip(
    axes,
    [M_zg, M_zp, M_gp],
    ["zeta vs GUE", "zeta vs Poisson", "GUE vs Poisson"],
    [n_zg, n_zp, n_gp]
):
    im = ax.imshow(M, aspect="auto", origin="lower", vmin=0, vmax=1)
    ax.set_title(f"{title} (n={n_used})")
    ax.set_xticks(np.arange(len(window_sizes)), window_sizes)
    ax.set_yticks(np.arange(len(feature_families)), feature_families)
    ax.set_xlabel("window size k")
    ax.set_ylabel("feature family")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="bootstrap mean score overlap")
plt.tight_layout()
plt.show()

## Overlap-gap phase diagrams

In [ ]:
gap_zg_zp = M_zg - M_zp
gap_zg_gp = M_zg - M_gp

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.8), sharey=True)

for ax, M, title in zip(
    axes,
    [gap_zg_zp, gap_zg_gp],
    ["zeta/GUE minus zeta/Poisson", "zeta/GUE minus GUE/Poisson"]
):
    im = ax.imshow(M, aspect="auto", origin="lower")
    ax.set_title(title)
    ax.set_xticks(np.arange(len(window_sizes)), window_sizes)
    ax.set_yticks(np.arange(len(feature_families)), feature_families)
    ax.set_xlabel("window size k")
    ax.set_ylabel("feature family")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="overlap gap")
plt.tight_layout()
plt.show()

## Height-resolved overlap diagrams (minimal features)

In [ ]:
H_zg, n_hzg = build_height_k_matrix(phase_results, "zeta_vs_GUE", "score_mean", fixed_feature="minimal", target_n=400)
H_zp, n_hzp = build_height_k_matrix(phase_results, "zeta_vs_Poisson", "score_mean", fixed_feature="minimal", target_n=400)
H_gp, n_hgp = build_height_k_matrix(phase_results, "GUE_vs_Poisson", "score_mean", fixed_feature="minimal", target_n=400)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharey=True)

for ax, M, title, n_used in zip(
    axes,
    [H_zg, H_zp, H_gp],
    ["zeta vs GUE", "zeta vs Poisson", "GUE vs Poisson"],
    [n_hzg, n_hzp, n_hgp]
):
    im = ax.imshow(M, aspect="auto", origin="lower", vmin=0, vmax=1)
    ax.set_title(f"{title} (n={n_used})")
    ax.set_xticks(np.arange(len(height_blocks)), [f"{a+1}-{b}" for a, b in height_blocks], rotation=20)
    ax.set_yticks(np.arange(len(window_sizes)), window_sizes)
    ax.set_xlabel("height block")
    ax.set_ylabel("window size k")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="bootstrap mean score overlap")
plt.tight_layout()
plt.show()

## Bootstrap-certainty masks

In [ ]:
S_zg, _ = build_feature_k_matrix(phase_results, "zeta_vs_GUE", "score_std", target_n=400)
S_zp, _ = build_feature_k_matrix(phase_results, "zeta_vs_Poisson", "score_std", target_n=400)
S_gp, _ = build_feature_k_matrix(phase_results, "GUE_vs_Poisson", "score_std", target_n=400)

std_threshold = 0.06
C_zg = (S_zg <= std_threshold).astype(float)
C_zp = (S_zp <= std_threshold).astype(float)
C_gp = (S_gp <= std_threshold).astype(float)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharey=True)

for ax, M, title in zip(
    axes,
    [C_zg, C_zp, C_gp],
    ["zeta vs GUE", "zeta vs Poisson", "GUE vs Poisson"]
):
    im = ax.imshow(M, aspect="auto", origin="lower", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks(np.arange(len(window_sizes)), window_sizes)
    ax.set_yticks(np.arange(len(feature_families)), feature_families)
    ax.set_xlabel("window size k")
    ax.set_ylabel("feature family")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="certainty mask (1 = std ≤ 0.06)")
plt.tight_layout()
plt.show()

## Error-bar slices across feature family

In [ ]:
plt.figure(figsize=(10.8, 4.8))
for task in ["zeta_vs_GUE", "zeta_vs_Poisson", "GUE_vs_Poisson"]:
    xs, means, lows, highs, used_n = aggregate_feature_slice(phase_results, task, "score", fixed_k=5, target_n=400)
    plt.errorbar(range(len(xs)), means, yerr=np.vstack([lows, highs]), marker="o", capsize=4, label=f"{task} (n={used_n})")
plt.xticks(range(len(feature_families)), feature_families, rotation=20)
plt.ylabel("bootstrap score overlap")
plt.title("Score overlap vs feature family (k=5)")
plt.legend()
plt.tight_layout()
plt.show()

## Invariant-region summary

In [ ]:
high_overlap_threshold = 0.5

invariant_summary = {
    "zeta_vs_GUE_high_overlap_fraction": float(np.mean(M_zg >= high_overlap_threshold)),
    "zeta_vs_Poisson_high_overlap_fraction": float(np.mean(M_zp >= high_overlap_threshold)),
    "GUE_vs_Poisson_high_overlap_fraction": float(np.mean(M_gp >= high_overlap_threshold)),
    "zeta_vs_GUE_certainty_fraction": float(np.mean(C_zg)),
    "zeta_vs_Poisson_certainty_fraction": float(np.mean(C_zp)),
    "GUE_vs_Poisson_certainty_fraction": float(np.mean(C_gp)),
}
invariant_summary

## Compact summary

In [ ]:
summary = {
    "n_records": len(phase_results),
    "available_sample_sizes": sorted(set(r["sample_size"] for r in phase_results)),
    "window_sizes": window_sizes,
    "feature_families": feature_families,
    "height_blocks": [f"{a+1}-{b}" for a, b in height_blocks],
    "n_boot": n_boot,
    "mean_overlap": {
        "zeta_vs_GUE": float(np.nanmean(M_zg)),
        "zeta_vs_Poisson": float(np.nanmean(M_zp)),
        "GUE_vs_Poisson": float(np.nanmean(M_gp)),
    },
    "mean_overlap_std": {
        "zeta_vs_GUE": float(np.nanmean(S_zg)),
        "zeta_vs_Poisson": float(np.nanmean(S_zp)),
        "GUE_vs_Poisson": float(np.nanmean(S_gp)),
    },
    "invariant_summary": invariant_summary,
}
summary

## Notes

- If zeta-vs-GUE occupies a larger high-overlap region than the Poisson tasks, and that region passes the bootstrap-certainty mask, then the overlap phase is both large and statistically stable.
- The gap plots help distinguish “high overlap in absolute terms” from “high overlap relative to the Poisson baselines.”
- This notebook is designed as a summary atlas rather than a maximal sweep.

## Next directions

A natural next notebook could do one of these:

1. build the same phase diagram for conditional windows  
2. add longer-range or multi-window features  
3. compare calibrated confidence maps rather than overlap alone  
4. test train/test mismatch across feature families